# DemandPulse

## Final System Evaluation & Production Readiness Review

**Notebook 07 — End-to-End Scientific Evaluation, Ablation Review and Production Recommendations**

This notebook is the final synthesis layer of the DemandPulse project.

It **does not retrain forecasting, causal, reconciliation, or inventory models**. Instead it reads the saved outputs produced by Notebooks 01–06, validates their availability and schema, and builds one evidence-grounded project review covering:

- point forecasting,
- probabilistic forecasting and calibration,
- causal promotion analysis,
- hierarchical reconciliation,
- inventory decision intelligence,
- monitoring and retraining strategy,
- computational/deployment footprint,
- ablation-style value assessment,
- production recommendations and research limitations.

> **Scientific rule:** every quantitative result in this notebook must come from previously executed DemandPulse artifacts or from arithmetic derived directly from those artifacts. Missing experiments remain missing; they are never filled with invented values.

## 1. Review questions

This notebook asks:

1. Did advanced point forecasting beat simple temporal baselines?
2. Are the probabilistic forecasts sharp and calibrated enough to support risk-sensitive decisions?
3. Is the promotion-effect evidence stable enough to be decision-relevant?
4. Does hierarchical reconciliation buy coherence without unacceptable accuracy loss?
5. Do forecast-driven and uncertainty-aware inventory policies improve the simulated decision metrics?
6. Are current monitoring signals compatible with deployment, continued observation, or retraining?
7. Which layers add measurable value, and which layers add complexity without demonstrated benefit?
8. What remains unproven before production?

## 2. Final architecture

```mermaid
flowchart TD
    A[Cloud Dataset<br/>FreshRetailNet-50K] --> B[Data Validation]
    B --> C[Feature Pipeline]
    C --> D[Forecasting]
    D --> D1[Point Forecasting]
    D --> D2[Probabilistic Forecasting]
    D --> D3[Hierarchical Forecasting]
    D1 --> E[Causal Promotion Engine]
    D2 --> E
    D3 --> E
    E --> F[Decision Optimization]
    F --> G[API / Serving Layer]
    G --> H[Monitoring]
    H --> I[Retraining Decision]
    I --> C
```

The API/serving layer is a **recommended production component**, not something trained or deployed in this notebook.

## 3. Environment

Only lightweight analysis/reporting packages are needed because this notebook consumes saved experiment outputs rather than fitting models.

In [ ]:
# %pip install -q "pandas>=2.0" "numpy>=1.24" "matplotlib>=3.7" "tabulate>=0.9"

In [ ]:
from __future__ import annotations

import hashlib
import json
import os
import platform
from pathlib import Path
from typing import Dict, Iterable, List, Mapping, Optional, Sequence, Tuple

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)

print("Python:", platform.python_version())
print("pandas:", pd.__version__)
print("numpy:", np.__version__)
print("matplotlib:", matplotlib.__version__)

## 4. Project configuration and output locations

The previous notebooks write their experiment tables under `reports/tables/`.  
This final notebook writes to `reports/final/` so that it **does not overwrite Notebook 01's own `final_model_comparison.csv`**.

In [ ]:
PROJECT_ROOT = Path(os.environ.get("DEMANDPULSE_PROJECT_ROOT", ".")).resolve()
SOURCE_TABLES_DIR = PROJECT_ROOT / "reports" / "tables"
MODELS_DIR = PROJECT_ROOT / "models"

FINAL_DIR = PROJECT_ROOT / "reports" / "final"
FINAL_FIGURES_DIR = FINAL_DIR / "figures"
for directory in [FINAL_DIR, FINAL_FIGURES_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

STRICT_ARTIFACT_REQUIREMENTS = True

OUTPUT_PATHS = {
    "final_project_scorecard": FINAL_DIR / "final_project_scorecard.csv",
    "final_model_comparison": FINAL_DIR / "final_model_comparison.csv",
    "business_impact_summary": FINAL_DIR / "business_impact_summary.csv",
    "ablation_results": FINAL_DIR / "ablation_results.csv",
    "final_research_findings": FINAL_DIR / "final_research_findings.md",
}

print("Project root:", PROJECT_ROOT)
print("Source tables:", SOURCE_TABLES_DIR)
print("Final outputs:", FINAL_DIR)

## 5. Previous-notebook artifact manifest

The final review is only valid if the upstream notebooks have been executed and their outputs saved.

The artifact audit intentionally fails early in strict mode when a scientifically required result is absent. This is preferable to silently substituting a different experiment or inventing a metric.

In [ ]:
ARTIFACT_MANIFEST = {
    # Notebook 01 — point forecasting
    "point_final": ("01", "final_model_comparison.csv", True),
    "point_backtest": ("01", "baseline_backtest_results.csv", True),
    "point_feature_importance": ("01", "feature_importance.csv", True),

    # Notebook 02 — probabilistic forecasting
    "prob_final": ("02", "probabilistic_final_metrics.csv", True),
    "prob_baseline": ("02", "probabilistic_baseline_comparison.csv", True),
    "prob_horizon": ("02", "probabilistic_metrics_by_horizon.csv", True),
    "conformal": ("02", "conformal_calibration.csv", True),
    "crossing": ("02", "quantile_crossing_diagnostics.csv", False),

    # Notebook 03 — causal promotion analysis
    "ate": ("03", "ate_estimates.csv", True),
    "att": ("03", "att_estimates.csv", True),
    "causal_sensitivity": ("03", "causal_sensitivity_analysis.csv", True),
    "cate_segments": ("03", "cate_segment_analysis.csv", True),
    "heterogeneous_effects": ("03", "heterogeneous_treatment_effects.csv", False),
    "placebo": ("03", "placebo_test_results.csv", True),

    # Notebook 04 — hierarchy
    "reconciliation": ("04", "reconciliation_results.csv", True),
    "hierarchy_levels": ("04", "hierarchy_level_metrics.csv", True),
    "coherence": ("04", "coherence_analysis.csv", True),

    # Notebook 05 — inventory
    "inventory_policy": ("05", "inventory_policy_comparison.csv", True),
    "inventory_simulation": ("05", "inventory_simulation_results.csv", True),
    "decision_recommendations": ("05", "decision_recommendations.csv", False),
    "inventory_costs": ("05", "inventory_cost_assumptions.csv", False),

    # Notebook 06 — monitoring
    "data_quality": ("06", "data_quality_monitoring.csv", True),
    "feature_drift": ("06", "feature_drift.csv", True),
    "forecast_monitoring": ("06", "forecast_performance_monitoring.csv", True),
    "calibration_monitoring": ("06", "calibration_monitoring.csv", True),
    "retraining": ("06", "retraining_decisions.csv", True),
}

audit_rows = []
for key, (notebook, filename, required) in ARTIFACT_MANIFEST.items():
    path = SOURCE_TABLES_DIR / filename
    audit_rows.append({
        "key": key,
        "source_notebook": notebook,
        "filename": filename,
        "required": required,
        "path": str(path),
        "exists": path.exists(),
        "size_bytes": path.stat().st_size if path.exists() else 0,
    })

artifact_audit = pd.DataFrame(audit_rows)
display(artifact_audit)

missing_required = artifact_audit.loc[
    artifact_audit["required"] & ~artifact_audit["exists"], ["source_notebook", "filename"]
]
if not missing_required.empty:
    message = (
        "Required previous-experiment artifacts are missing.\n"
        "Execute the indicated DemandPulse notebooks before running the final review:\n"
        + missing_required.to_string(index=False)
    )
    if STRICT_ARTIFACT_REQUIREMENTS:
        raise FileNotFoundError(message)
    print("WARNING:", message)

## 6. Artifact loading and schema validation

Each source table is loaded independently from disk. No Python variable from an earlier notebook kernel is reused.

In [ ]:
def load_csv_artifact(key: str) -> pd.DataFrame:
    notebook, filename, required = ARTIFACT_MANIFEST[key]
    path = SOURCE_TABLES_DIR / filename
    if not path.exists():
        if required:
            raise FileNotFoundError(f"Notebook {notebook} artifact missing: {path}")
        return pd.DataFrame()
    frame = pd.read_csv(path)
    if frame.empty:
        raise ValueError(f"Artifact is empty: {path}")
    return frame


def require_columns(df: pd.DataFrame, columns: Sequence[str], artifact_name: str) -> None:
    missing = sorted(set(columns) - set(df.columns))
    if missing:
        raise ValueError(f"{artifact_name} is missing required columns: {missing}")


tables: Dict[str, pd.DataFrame] = {
    key: load_csv_artifact(key)
    for key in ARTIFACT_MANIFEST
    if (SOURCE_TABLES_DIR / ARTIFACT_MANIFEST[key][1]).exists()
}

# Core schema checks.
require_columns(tables["point_final"], ["model", "MAE", "RMSE", "WAPE", "Bias", "observations"], "Notebook 01 final comparison")
require_columns(tables["prob_final"], ["model", "mae", "wape", "mean_pinball", "coverage", "mean_width", "interval_score", "observations"], "Notebook 02 probabilistic metrics")
require_columns(tables["ate"], ["Method", "ATE", "CI_Lower", "CI_Upper"], "Notebook 03 ATE")
require_columns(tables["att"], ["Method", "ATT", "CI_Lower", "CI_Upper"], "Notebook 03 ATT")
require_columns(tables["reconciliation"], ["method", "MAE", "RMSE", "WAPE", "Bias", "normalized_coherence_error"], "Notebook 04 reconciliation")
require_columns(tables["inventory_policy"], ["policy", "realized_mean_cost", "stockout_rate", "service_level_fill_rate", "simulated_profit"], "Notebook 05 inventory policy")
require_columns(tables["retraining"], ["monitoring_period", "monitoring_status", "decision", "reason"], "Notebook 06 retraining")

print(f"Loaded {len(tables)} previous experiment tables.")

## 7. Provenance fingerprints

A reproducible review should be able to identify exactly which upstream outputs were consumed.  
SHA-256 hashes below are fingerprints of the saved CSV artifacts, not model-performance metrics.

In [ ]:
def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as f:
        while True:
            block = f.read(chunk_size)
            if not block:
                break
            digest.update(block)
    return digest.hexdigest()


provenance_rows = []
for row in artifact_audit.loc[artifact_audit["exists"]].itertuples(index=False):
    path = Path(row.path)
    provenance_rows.append({
        "source_notebook": row.source_notebook,
        "filename": row.filename,
        "size_bytes": row.size_bytes,
        "sha256": sha256_file(path),
    })

provenance = pd.DataFrame(provenance_rows)
display(provenance)

# Point Forecasting Review

Notebook 01 compared naive temporal baselines and global LightGBM on chronological evaluation.

The final review does not rank the oracle-weather experiment as production-realistic because that experiment used realized target-date weather as an upper-bound study.

In [ ]:
point_final = tables["point_final"].copy()
display(point_final.sort_values("WAPE"))

core_point = point_final.loc[point_final["model"].eq("LightGBM")]
seasonal = point_final.loc[point_final["model"].eq("Seasonal Naive")]
if core_point.empty or seasonal.empty:
    raise ValueError("Notebook 01 must contain both LightGBM and Seasonal Naive rows.")

point_lgbm = core_point.iloc[0]
point_seasonal = seasonal.iloc[0]
point_wape_gain_pct = (
    100 * (point_seasonal["WAPE"] - point_lgbm["WAPE"]) / point_seasonal["WAPE"]
    if point_seasonal["WAPE"] != 0 else np.nan
)

print(f"LightGBM official WAPE: {point_lgbm['WAPE']:.6f}")
print(f"Seasonal Naive official WAPE: {point_seasonal['WAPE']:.6f}")
print(f"Relative WAPE improvement of LightGBM vs Seasonal Naive: {point_wape_gain_pct:.2f}%")

# Probabilistic Forecasting Review

Notebook 02 evaluates conditional P10/P50/P90 forecasts, raw P10–P90 intervals, and conformalized intervals.  
Coverage and interval width must be interpreted together: higher coverage can be purchased by making intervals wider.

In [ ]:
prob_final = tables["prob_final"].copy()
display(prob_final)

raw_prob = prob_final.loc[prob_final["model"].eq("Raw LightGBM Quantile")]
conf_prob = prob_final.loc[prob_final["model"].eq("Conformalized LightGBM Quantile")]
prob_baseline = prob_final.loc[prob_final["model"].eq("Seasonal-Naive probabilistic baseline")]

if raw_prob.empty or conf_prob.empty:
    raise ValueError("Notebook 02 must contain raw and conformal LightGBM quantile rows.")

raw_prob = raw_prob.iloc[0]
conf_prob = conf_prob.iloc[0]
coverage_change = conf_prob["coverage"] - raw_prob["coverage"]
width_change = conf_prob["mean_width"] - raw_prob["mean_width"]

print(f"Raw interval coverage: {raw_prob['coverage']:.2%}")
print(f"Conformal interval coverage: {conf_prob['coverage']:.2%}")
print(f"Coverage change: {coverage_change:+.2%}")
print(f"Mean interval width change: {width_change:+.6f}")

In [ ]:
prob_horizon = tables["prob_horizon"].copy()
display(prob_horizon.sort_values("horizon"))

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(prob_horizon["horizon"], prob_horizon["wape"], marker="o", label="P50 WAPE")
ax.set_title("Probabilistic Forecast Accuracy by Horizon")
ax.set_xlabel("Forecast horizon (days)")
ax.set_ylabel("WAPE")
ax.legend()
fig.tight_layout()
fig.savefig(FINAL_FIGURES_DIR / "probabilistic_wape_by_horizon.png", dpi=150, bbox_inches="tight")
plt.show()

# Causal Promotion Evidence Review

The causal layer is observational. ATE/ATT/CATE estimates require consistency, positivity, temporal ordering, no major unmeasured confounding, and suitable model specification.

The final review therefore reports:

- cross-fitted AIPW ATE,
- doubly robust ATT,
- sensitivity range,
- placebo/falsification evidence,
- CATE segment heterogeneity,

without treating any one estimate as experimental ground truth.

In [ ]:
ate = tables["ate"].copy()
att = tables["att"].copy()
sensitivity = tables["causal_sensitivity"].copy()
placebo = tables["placebo"].copy()

display(ate)
display(att)
display(sensitivity)
display(placebo)

aipw_rows = ate.loc[ate["Method"].astype(str).str.upper().eq("AIPW")]
if aipw_rows.empty:
    raise ValueError("Notebook 03 ATE table must contain an AIPW estimate.")
aipw = aipw_rows.iloc[0]
att_main = att.iloc[0]

sens_values = pd.to_numeric(sensitivity["ATE_AIPW"], errors="coerce").dropna()
sens_min = float(sens_values.min()) if len(sens_values) else np.nan
sens_max = float(sens_values.max()) if len(sens_values) else np.nan
sens_range = sens_max - sens_min if np.isfinite(sens_min) and np.isfinite(sens_max) else np.nan

placebo_main = placebo.iloc[0]
placebo_raw = placebo_main.get("residual_confounding_concern", False)
if isinstance(placebo_raw, str):
    placebo_concern = placebo_raw.strip().lower() in {"true", "1", "yes", "y"}
else:
    placebo_concern = bool(placebo_raw)

print(f"AIPW ATE: {aipw['ATE']:.6f} [{aipw['CI_Lower']:.6f}, {aipw['CI_Upper']:.6f}]")
print(f"ATT: {att_main['ATT']:.6f} [{att_main['CI_Lower']:.6f}, {att_main['CI_Upper']:.6f}]")
print(f"Sensitivity ATE range: [{sens_min:.6f}, {sens_max:.6f}]")
print(f"Placebo residual-confounding concern: {placebo_concern}")

In [ ]:
cate_segments = tables["cate_segments"].copy()
require_columns(cate_segments, ["segment_type", "segment", "sample_size", "mean_cate"], "CATE segment analysis")

largest_cate = cate_segments.sort_values("mean_cate", ascending=False).head(10)
lowest_cate = cate_segments.sort_values("mean_cate", ascending=True).head(10)

print("Highest estimated CATE segments")
display(largest_cate)
print("Lowest estimated CATE segments")
display(lowest_cate)

# Hierarchical Forecasting & Reconciliation Review

The hierarchy compares independent forecasts with Bottom-Up, Top-Down, Middle-Out and a practical diagonal-W MinT-style reconciler.

A reconciled system is useful only if the gain in accounting consistency is worth any accuracy redistribution across hierarchy levels.

In [ ]:
reconciliation = tables["reconciliation"].copy()
level_metrics = tables["hierarchy_levels"].copy()

display(reconciliation.sort_values("WAPE"))

unrec = reconciliation.loc[reconciliation["method"].eq("Unreconciled")]
rec_candidates = reconciliation.loc[~reconciliation["method"].eq("Unreconciled")]
if unrec.empty or rec_candidates.empty:
    raise ValueError("Notebook 04 reconciliation results are incomplete.")

unrec = unrec.iloc[0]
best_rec = rec_candidates.sort_values("WAPE").iloc[0]

hier_wape_gain_pct = (
    100 * (unrec["WAPE"] - best_rec["WAPE"]) / unrec["WAPE"]
    if unrec["WAPE"] != 0 else np.nan
)
coherence_reduction_pct = (
    100 * (unrec["normalized_coherence_error"] - best_rec["normalized_coherence_error"])
    / unrec["normalized_coherence_error"]
    if unrec["normalized_coherence_error"] != 0 else np.nan
)

print(f"Best reconciled method: {best_rec['method']}")
print(f"WAPE: {best_rec['WAPE']:.6f} vs unreconciled {unrec['WAPE']:.6f}")
print(f"Relative WAPE improvement: {hier_wape_gain_pct:.2f}%")
print(f"Normalized coherence error reduction: {coherence_reduction_pct:.2f}%")

In [ ]:
best_method = str(best_rec["method"])
base_level = level_metrics.loc[level_metrics["method"].eq("Unreconciled"), ["level", "WAPE"]].rename(
    columns={"WAPE": "unreconciled_WAPE"}
)
best_level = level_metrics.loc[level_metrics["method"].eq(best_method), ["level", "WAPE", "coherence_error"]].rename(
    columns={"WAPE": "reconciled_WAPE"}
)
level_tradeoff = best_level.merge(base_level, on="level", how="inner")
level_tradeoff["WAPE_change"] = level_tradeoff["reconciled_WAPE"] - level_tradeoff["unreconciled_WAPE"]
level_tradeoff["accuracy_improved"] = level_tradeoff["WAPE_change"] < 0
display(level_tradeoff)

fig, ax = plt.subplots(figsize=(9, 4.5))
x = np.arange(len(level_tradeoff))
width = 0.38
ax.bar(x - width/2, level_tradeoff["unreconciled_WAPE"], width, label="Unreconciled")
ax.bar(x + width/2, level_tradeoff["reconciled_WAPE"], width, label=best_method)
ax.set_xticks(x)
ax.set_xticklabels(level_tradeoff["level"].astype(str), rotation=25)
ax.set_ylabel("WAPE")
ax.set_title("Accuracy Trade-off by Hierarchy Level")
ax.legend()
fig.tight_layout()
fig.savefig(FINAL_FIGURES_DIR / "hierarchy_accuracy_tradeoff.png", dpi=150, bbox_inches="tight")
plt.show()

# Inventory Decision Intelligence Review

Notebook 05 used configurable **simulation assumptions**, not claimed retailer financials.  
Accordingly, cost/profit findings are scenario-dependent decision evidence rather than audited business P&L.

In [ ]:
inventory = tables["inventory_policy"].copy()
inventory_sim = tables["inventory_simulation"].copy()

display(inventory.sort_values("realized_mean_cost"))

hist_policy = inventory.loc[inventory["policy"].eq("Historical Average")]
if hist_policy.empty:
    raise ValueError("Historical Average policy is required for business-value comparison.")
hist_policy = hist_policy.iloc[0]
best_inventory = inventory.sort_values("realized_mean_cost").iloc[0]

inventory_cost_gain_pct = (
    100 * (hist_policy["realized_mean_cost"] - best_inventory["realized_mean_cost"])
    / hist_policy["realized_mean_cost"]
    if hist_policy["realized_mean_cost"] != 0 else np.nan
)
inventory_profit_delta = best_inventory["simulated_profit"] - hist_policy["simulated_profit"]

print(f"Lowest realized mean cost policy: {best_inventory['policy']}")
print(f"Relative cost reduction vs Historical Average: {inventory_cost_gain_pct:.2f}%")
print(f"Simulated profit delta vs Historical Average: {inventory_profit_delta:+.6f}")

In [ ]:
# Promotion-scenario causal adjustment comparison, if Notebook 05 produced both variants.
promo_sim = inventory_sim.loc[inventory_sim["scenario"].astype(str).str.lower().eq("promotion")].copy()
causal_business_evidence = pd.DataFrame()

if {"Optimized Decision", "Optimized No Causal"}.issubset(set(promo_sim["policy"])):
    causal_full = promo_sim.loc[promo_sim["policy"].eq("Optimized Decision")].iloc[0]
    causal_none = promo_sim.loc[promo_sim["policy"].eq("Optimized No Causal")].iloc[0]
    causal_business_evidence = pd.DataFrame([{
        "scenario": "Promotion",
        "causal_policy": "Optimized Decision",
        "no_causal_policy": "Optimized No Causal",
        "expected_cost_with_causal": causal_full["expected_cost"],
        "expected_cost_without_causal": causal_none["expected_cost"],
        "cost_delta": causal_full["expected_cost"] - causal_none["expected_cost"],
        "profit_with_causal": causal_full["simulated_profit"],
        "profit_without_causal": causal_none["simulated_profit"],
        "profit_delta": causal_full["simulated_profit"] - causal_none["simulated_profit"],
        "service_with_causal": causal_full["service_level"],
        "service_without_causal": causal_none["service_level"],
    }])
    display(causal_business_evidence)
else:
    print("Promotion simulation does not contain both causal and no-causal optimized variants.")

# Monitoring & Retraining Review

Notebook 06 separates data incidents, drift, degradation, calibration failure and retraining decisions.

Drift alone is not sufficient evidence to retrain.

In [ ]:
retraining = tables["retraining"].copy()
feature_drift = tables["feature_drift"].copy()
forecast_monitoring = tables["forecast_monitoring"].copy()
calibration_monitoring = tables["calibration_monitoring"].copy()
data_quality = tables["data_quality"].copy()

retraining["monitoring_period"] = pd.to_datetime(retraining["monitoring_period"], errors="coerce")
latest_monitoring = retraining.sort_values("monitoring_period").iloc[-1]

max_psi = pd.to_numeric(feature_drift.get("psi"), errors="coerce").max()
max_ks = pd.to_numeric(feature_drift.get("ks_statistic"), errors="coerce").max() if "ks_statistic" in feature_drift else np.nan
retrain_recommended_count = int(retraining["decision"].eq("RETRAIN_RECOMMENDED").sum())

print("Latest monitoring status:", latest_monitoring["monitoring_status"])
print("Latest retraining decision:", latest_monitoring["decision"])
print("Reason:", latest_monitoring["reason"])
print("Maximum observed PSI:", max_psi)
print("Maximum observed KS statistic:", max_ks)
print("Periods with RETRAIN_RECOMMENDED:", retrain_recommended_count)

display(retraining.tail(10))

# Computational Cost & Scalability Evidence

The previous notebooks did not require a standardized wall-clock/CPU/GPU benchmark artifact, so this final review does **not invent runtime or cloud-cost numbers**.

Instead it reports what can be measured directly from saved production artifacts:

- number of trained model files,
- model artifact footprint,
- architecture/model multiplicity.

This is a deployment-footprint proxy, not a training-cost benchmark.

In [ ]:
def summarize_model_footprint(models_root: Path) -> pd.DataFrame:
    groups = {
        "Point": [models_root / "demandpulse_lgbm.txt"],
        "Probabilistic": list((models_root / "quantile").glob("h*_q*.txt")) if (models_root / "quantile").exists() else [],
        "Causal nuisance": list((models_root / "causal").glob("*.txt")) if (models_root / "causal").exists() else [],
        "Hierarchical base": list((models_root / "hierarchical").glob("*.txt")) if (models_root / "hierarchical").exists() else [],
    }
    rows = []
    for stage, paths in groups.items():
        existing = [p for p in paths if p.exists()]
        rows.append({
            "stage": stage,
            "model_files": len(existing),
            "total_bytes": sum(p.stat().st_size for p in existing),
            "total_mb": sum(p.stat().st_size for p in existing) / (1024 ** 2),
            "runtime_benchmark_available": False,
        })
    return pd.DataFrame(rows)

model_footprint = summarize_model_footprint(MODELS_DIR)
display(model_footprint)
print("Total model files:", int(model_footprint["model_files"].sum()))
print("Total model footprint (MB):", float(model_footprint["total_mb"].sum()))

# Unified Model Comparison

Metrics from different notebooks are not automatically apples-to-apples.

This table therefore records an explicit `comparison_group`:

- Notebook 01 official point-eval rows,
- Notebook 02 probabilistic common-mask rows,
- Notebook 04 hierarchical common-grid rows.

Models should be ranked **within** their comparison group, not across incompatible observation sets.

In [ ]:
comparison_rows: List[dict] = []

for row in point_final.itertuples(index=False):
    comparison_rows.append({
        "stage": "Point",
        "model": row.model,
        "comparison_group": "Notebook01_official_eval",
        "MAE": row.MAE,
        "RMSE": row.RMSE,
        "WAPE": row.WAPE,
        "Bias": row.Bias,
        "mean_pinball": np.nan,
        "coverage": np.nan,
        "mean_width": np.nan,
        "interval_score": np.nan,
        "coherence_error": np.nan,
        "observations": getattr(row, "observations", np.nan),
        "source_artifact": "reports/tables/final_model_comparison.csv",
    })

for row in prob_final.itertuples(index=False):
    comparison_rows.append({
        "stage": "Probabilistic",
        "model": row.model,
        "comparison_group": "Notebook02_probabilistic_common_mask",
        "MAE": getattr(row, "mae", np.nan),
        "RMSE": getattr(row, "rmse", np.nan),
        "WAPE": getattr(row, "wape", np.nan),
        "Bias": getattr(row, "bias", np.nan),
        "mean_pinball": getattr(row, "mean_pinball", np.nan),
        "coverage": getattr(row, "coverage", np.nan),
        "mean_width": getattr(row, "mean_width", np.nan),
        "interval_score": getattr(row, "interval_score", np.nan),
        "coherence_error": np.nan,
        "observations": getattr(row, "observations", np.nan),
        "source_artifact": "reports/tables/probabilistic_final_metrics.csv",
    })

for row in reconciliation.itertuples(index=False):
    comparison_rows.append({
        "stage": "Hierarchical",
        "model": row.method,
        "comparison_group": "Notebook04_hierarchy_common_grid",
        "MAE": row.MAE,
        "RMSE": row.RMSE,
        "WAPE": row.WAPE,
        "Bias": row.Bias,
        "mean_pinball": np.nan,
        "coverage": np.nan,
        "mean_width": np.nan,
        "interval_score": np.nan,
        "coherence_error": row.normalized_coherence_error,
        "observations": getattr(row, "observations", np.nan),
        "source_artifact": "reports/tables/reconciliation_results.csv",
    })

final_model_comparison = pd.DataFrame(comparison_rows)
display(final_model_comparison)

# Ablation-Style Complexity Review

An ablation is only treated as quantitative evidence when the project contains a valid executed comparator.

### Evidence classes

- **EXECUTED:** direct saved comparator exists.
- **PARTIAL:** a decision-level proxy exists, but it does not isolate one modeling change perfectly.
- **NOT_EVALUATED:** the required counterfactual experiment was never run.

This prevents an attractive but scientifically invalid “ablation” from being inferred from unrelated tables.

In [ ]:
def relative_benefit_lower_is_better(full: float, ablated: float) -> float:
    if not np.isfinite(full) or not np.isfinite(ablated) or ablated == 0:
        return np.nan
    return 100 * (ablated - full) / abs(ablated)


ablation_rows: List[dict] = []

# 1. Advanced ML removed -> Seasonal Naive only.
ablation_rows.append({
    "component_removed": "Advanced ML (Seasonal Naive only)",
    "full_system_variant": "LightGBM",
    "ablated_variant": "Seasonal Naive",
    "primary_metric": "WAPE",
    "lower_is_better": True,
    "full_value": float(point_lgbm["WAPE"]),
    "ablated_value": float(point_seasonal["WAPE"]),
    "benefit_of_full_system_pct": relative_benefit_lower_is_better(float(point_lgbm["WAPE"]), float(point_seasonal["WAPE"])),
    "evidence_status": "EXECUTED",
    "source": "Notebook 01 official eval",
    "interpretation": "Positive benefit means LightGBM reduced WAPE relative to Seasonal Naive on the same Notebook 01 evaluation.",
})

# 2. Hierarchy reconciliation removed.
ablation_rows.append({
    "component_removed": "Hierarchy reconciliation",
    "full_system_variant": str(best_rec["method"]),
    "ablated_variant": "Unreconciled",
    "primary_metric": "WAPE",
    "lower_is_better": True,
    "full_value": float(best_rec["WAPE"]),
    "ablated_value": float(unrec["WAPE"]),
    "benefit_of_full_system_pct": relative_benefit_lower_is_better(float(best_rec["WAPE"]), float(unrec["WAPE"])),
    "evidence_status": "EXECUTED",
    "source": "Notebook 04 common hierarchy grid",
    "interpretation": f"Reconciliation also changed normalized coherence error from {unrec['normalized_coherence_error']:.6g} to {best_rec['normalized_coherence_error']:.6g}.",
})

# 3. Probabilistic decision inputs removed.
point_policy = inventory.loc[inventory["policy"].eq("Point Forecast")]
prob_simple = inventory.loc[inventory["policy"].isin(["P50", "P90 Conservative"])]
if not point_policy.empty and not prob_simple.empty:
    point_policy = point_policy.iloc[0]
    best_prob_policy = prob_simple.sort_values("realized_mean_cost").iloc[0]
    ablation_rows.append({
        "component_removed": "Probabilistic forecasting inputs",
        "full_system_variant": str(best_prob_policy["policy"]),
        "ablated_variant": "Point Forecast",
        "primary_metric": "realized_mean_cost",
        "lower_is_better": True,
        "full_value": float(best_prob_policy["realized_mean_cost"]),
        "ablated_value": float(point_policy["realized_mean_cost"]),
        "benefit_of_full_system_pct": relative_benefit_lower_is_better(float(best_prob_policy["realized_mean_cost"]), float(point_policy["realized_mean_cost"])),
        "evidence_status": "PARTIAL",
        "source": "Notebook 05 realized policy comparison",
        "interpretation": "Decision-level proxy only: P50/P90 policy comparison does not perfectly isolate the statistical value of the entire probabilistic model.",
    })

# 4. Causal adjustment removed.
if not causal_business_evidence.empty:
    r = causal_business_evidence.iloc[0]
    ablation_rows.append({
        "component_removed": "Causal promotion adjustment",
        "full_system_variant": "Optimized Decision",
        "ablated_variant": "Optimized No Causal",
        "primary_metric": "promotion_scenario_expected_cost",
        "lower_is_better": True,
        "full_value": float(r["expected_cost_with_causal"]),
        "ablated_value": float(r["expected_cost_without_causal"]),
        "benefit_of_full_system_pct": relative_benefit_lower_is_better(float(r["expected_cost_with_causal"]), float(r["expected_cost_without_causal"])),
        "evidence_status": "EXECUTED",
        "source": "Notebook 05 promotion Monte Carlo scenario",
        "interpretation": f"Simulated profit delta with causal uncertainty was {r['profit_delta']:+.6f}; financial values depend on configured simulation assumptions.",
    })
else:
    ablation_rows.append({
        "component_removed": "Causal promotion adjustment",
        "full_system_variant": "Optimized Decision",
        "ablated_variant": "Optimized No Causal",
        "primary_metric": "promotion_scenario_expected_cost",
        "lower_is_better": True,
        "full_value": np.nan,
        "ablated_value": np.nan,
        "benefit_of_full_system_pct": np.nan,
        "evidence_status": "NOT_EVALUATED",
        "source": "Notebook 05 artifact did not contain both variants",
        "interpretation": "No valid saved causal/no-causal decision comparison was available.",
    })

# 5. Promotion features removed.
# No earlier notebook trained a matched forecasting model with activity/promotion predictors removed.
ablation_rows.append({
    "component_removed": "Promotion features",
    "full_system_variant": "Current forecasting feature set",
    "ablated_variant": "Forecast model without promotion features",
    "primary_metric": "WAPE",
    "lower_is_better": True,
    "full_value": np.nan,
    "ablated_value": np.nan,
    "benefit_of_full_system_pct": np.nan,
    "evidence_status": "NOT_EVALUATED",
    "source": "No matched no-promotion-feature forecasting artifact in Notebooks 01–06",
    "interpretation": "A true forecasting-feature ablation would require a controlled rerun; causal treatment estimates cannot substitute for this predictive ablation.",
})

ablation_results = pd.DataFrame(ablation_rows)
display(ablation_results)

## Conformal calibration as an additional complexity trade-off

Conformalization is not one of the requested removal ablations, but it is a useful example of added complexity with an explicit measurable price: empirical coverage may improve while interval width grows.

In [ ]:
conformal_tradeoff = pd.DataFrame([{
    "component": "Conformal calibration",
    "raw_coverage": raw_prob["coverage"],
    "conformal_coverage": conf_prob["coverage"],
    "coverage_change": coverage_change,
    "raw_mean_width": raw_prob["mean_width"],
    "conformal_mean_width": conf_prob["mean_width"],
    "width_change": width_change,
    "raw_interval_score": raw_prob["interval_score"],
    "conformal_interval_score": conf_prob["interval_score"],
}])
display(conformal_tradeoff)

# Business Impact Summary

The table below combines only **derived comparisons supported by previous experiment outputs**.  
Inventory monetary quantities remain simulation outputs under Notebook 05's configurable assumptions.

In [ ]:
business_rows = [
    {
        "area": "Point forecasting",
        "comparison": "LightGBM vs Seasonal Naive",
        "metric": "WAPE relative improvement (%)",
        "value": point_wape_gain_pct,
        "secondary_metric": "LightGBM Bias",
        "secondary_value": float(point_lgbm["Bias"]),
        "evidence": "Notebook 01 official eval",
    },
    {
        "area": "Probabilistic calibration",
        "comparison": "Conformal vs raw P10-P90",
        "metric": "coverage change",
        "value": coverage_change,
        "secondary_metric": "mean width change",
        "secondary_value": width_change,
        "evidence": "Notebook 02 common evaluation mask",
    },
    {
        "area": "Hierarchy reconciliation",
        "comparison": f"{best_rec['method']} vs Unreconciled",
        "metric": "WAPE relative improvement (%)",
        "value": hier_wape_gain_pct,
        "secondary_metric": "coherence error reduction (%)",
        "secondary_value": coherence_reduction_pct,
        "evidence": "Notebook 04 common hierarchy grid",
    },
    {
        "area": "Inventory",
        "comparison": f"{best_inventory['policy']} vs Historical Average",
        "metric": "realized mean cost reduction (%)",
        "value": inventory_cost_gain_pct,
        "secondary_metric": "simulated profit delta",
        "secondary_value": inventory_profit_delta,
        "evidence": "Notebook 05 configured-cost evaluation",
    },
]

if not causal_business_evidence.empty:
    r = causal_business_evidence.iloc[0]
    business_rows.append({
        "area": "Causal decision adjustment",
        "comparison": "Optimized causal-aware vs no-causal promotion scenario",
        "metric": "expected cost delta",
        "value": float(r["cost_delta"]),
        "secondary_metric": "simulated profit delta",
        "secondary_value": float(r["profit_delta"]),
        "evidence": "Notebook 05 Monte Carlo promotion scenario",
    })

business_impact_summary = pd.DataFrame(business_rows)
display(business_impact_summary)

# Final System Scorecard

The scorecard deliberately avoids a fabricated single 0–100 score. A composite grade would require arbitrary weights across error, calibration, causal validity, business simulation and monitoring.

Instead each system dimension is reported with its most decision-relevant executed evidence and an explicit evidence status.

In [ ]:
total_model_count = int(model_footprint["model_files"].sum())
total_model_mb = float(model_footprint["total_mb"].sum())

monitor_status = str(latest_monitoring["monitoring_status"])
monitor_decision = str(latest_monitoring["decision"])

causal_ci_excludes_zero = not (float(aipw["CI_Lower"]) <= 0 <= float(aipw["CI_Upper"]))

scorecard_rows = [
    {
        "dimension": "Model performance",
        "primary_evidence": "LightGBM WAPE",
        "value": float(point_lgbm["WAPE"]),
        "secondary_evidence": "Relative WAPE gain vs Seasonal Naive (%)",
        "secondary_value": point_wape_gain_pct,
        "status": "MEASURED",
        "limitation": "Notebook 01 evaluation scope; oracle weather excluded from production ranking.",
    },
    {
        "dimension": "Uncertainty quality",
        "primary_evidence": "Conformal P10-P90 coverage",
        "value": float(conf_prob["coverage"]),
        "secondary_evidence": "Conformal mean interval width",
        "secondary_value": float(conf_prob["mean_width"]),
        "status": "MEASURED",
        "limitation": "Only P10/P50/P90 are modeled; intervals do not define a complete predictive distribution.",
    },
    {
        "dimension": "Causal evidence",
        "primary_evidence": "Cross-fitted AIPW ATE",
        "value": float(aipw["ATE"]),
        "secondary_evidence": "ATE CI excludes zero",
        "secondary_value": float(causal_ci_excludes_zero),
        "status": "CAUTION" if placebo_concern else "MEASURED",
        "limitation": f"Observational assumptions required; placebo concern={placebo_concern}; sensitivity range={sens_range:.6f}.",
    },
    {
        "dimension": "Hierarchy consistency",
        "primary_evidence": f"{best_rec['method']} normalized coherence error",
        "value": float(best_rec["normalized_coherence_error"]),
        "secondary_evidence": "Coherence error reduction vs unreconciled (%)",
        "secondary_value": coherence_reduction_pct,
        "status": "MEASURED",
        "limitation": "Best reconciler by overall WAPE may improve some hierarchy levels and worsen others.",
    },
    {
        "dimension": "Business value",
        "primary_evidence": "Best policy realized mean cost",
        "value": float(best_inventory["realized_mean_cost"]),
        "secondary_evidence": "Cost reduction vs historical average (%)",
        "secondary_value": inventory_cost_gain_pct,
        "status": "SIMULATION",
        "limitation": "Financial costs/prices are configurable simulation assumptions, not retailer accounting data.",
    },
    {
        "dimension": "Monitoring readiness",
        "primary_evidence": "Latest monitoring status",
        "value": np.nan,
        "secondary_evidence": "Retraining decision",
        "secondary_value": np.nan,
        "status": monitor_status,
        "limitation": f"Latest decision={monitor_decision}; drift alone does not trigger retraining.",
    },
    {
        "dimension": "Computational footprint",
        "primary_evidence": "Saved model file count",
        "value": float(total_model_count),
        "secondary_evidence": "Saved model footprint MB",
        "secondary_value": total_model_mb,
        "status": "MEASURED",
        "limitation": "Wall-clock training latency, serving latency and cloud cost were not standardized/logged.",
    },
    {
        "dimension": "Scalability",
        "primary_evidence": "Full production-scale benchmark",
        "value": np.nan,
        "secondary_evidence": "Runtime benchmark",
        "secondary_value": np.nan,
        "status": "EVIDENCE_GAP",
        "limitation": "The project uses streaming/global-model designs, but this final review has no standardized full-data throughput/latency artifact to quantify production scalability.",
    },
]

final_project_scorecard = pd.DataFrame(scorecard_rows)
display(final_project_scorecard)

# Failure Analysis

Complexity is not automatically valuable. This section identifies where the saved experiments show trade-offs, fragility, or missing evidence.

In [ ]:
failure_rows: List[dict] = []

# Advanced ML can fail to beat seasonal naive.
if np.isfinite(point_wape_gain_pct) and point_wape_gain_pct <= 0:
    failure_rows.append({
        "component": "Point LightGBM",
        "issue": "Did not improve WAPE over Seasonal Naive",
        "evidence": point_wape_gain_pct,
    })

# Hierarchy-level accuracy degradation after best reconciliation.
worse_levels = level_tradeoff.loc[level_tradeoff["WAPE_change"] > 0]
for r in worse_levels.itertuples(index=False):
    failure_rows.append({
        "component": f"Hierarchy reconciliation — {r.level}",
        "issue": "Best overall reconciler worsened level WAPE",
        "evidence": float(r.WAPE_change),
    })

# Calibration price.
if np.isfinite(width_change) and width_change > 0:
    failure_rows.append({
        "component": "Conformal calibration",
        "issue": "Coverage adjustment widened intervals",
        "evidence": float(width_change),
    })

# Causal fragility.
if placebo_concern:
    failure_rows.append({
        "component": "Causal promotion engine",
        "issue": "Placebo test raised residual-confounding/misspecification concern",
        "evidence": float(placebo_main["AIPW_placebo_effect"]),
    })

if np.isfinite(sens_min) and np.isfinite(sens_max) and sens_min <= 0 <= sens_max:
    failure_rows.append({
        "component": "Causal promotion engine",
        "issue": "Sensitivity analyses span zero",
        "evidence": float(sens_range),
    })

# Monitoring.
if monitor_status in {"YELLOW", "RED"}:
    failure_rows.append({
        "component": "Production monitoring",
        "issue": f"Latest status is {monitor_status}",
        "evidence": monitor_decision,
    })

# Explicit missing ablation.
failure_rows.append({
    "component": "Promotion-feature forecasting ablation",
    "issue": "No matched executed no-promotion-feature forecasting experiment",
    "evidence": "NOT_EVALUATED",
})

failure_analysis = pd.DataFrame(failure_rows)
display(failure_analysis)

# Production Recommendations

Recommendations below are evidence-governed rather than automatic deployment approval.

1. **Preserve temporal evaluation contracts.** Keep official future holdouts and rolling-origin backtests separated from model selection.
2. **Serve point and probabilistic forecasts together.** Inventory decisions need both expected/median demand and uncertainty.
3. **Use reconciled hierarchy outputs where accounting coherence is operationally required**, but monitor level-specific accuracy after reconciliation.
4. **Treat causal promotion estimates as an advisory signal with uncertainty**, not a deterministic uplift.
5. **Do not convert simulation costs into claimed financial ROI** until real retailer cost/service parameters are supplied.
6. **Gate retraining on persistent degradation + sufficient new labels + acceptable data quality**, consistent with Notebook 06.
7. **Add shadow/canary serving before full rollout**, with rollback tied to accuracy, calibration, service-level and data-quality thresholds.
8. **Benchmark production scale explicitly**: streaming throughput, feature latency, prediction latency, memory, CPU/GPU cost and model-loading footprint.
9. **Run the missing promotion-feature forecasting ablation** before claiming the predictive value of promotion variables.
10. **Separate decision governance for cold-start and stockout-censored series**, where historical features or observed demand are structurally limited.

# Future Research

High-value next research directions include:

- probabilistic hierarchical reconciliation rather than point-only reconciliation,
- true grouped/multi-dimensional hierarchy reconciliation,
- learned or distributional inventory optimization with calibrated service-level objectives,
- treatment-effect estimation with stronger identification strategies or experimental data,
- explicit lost-sales / latent-demand modeling under stockouts,
- cold-start forecasting architectures,
- dynamic pricing/promotion policy optimization,
- standardized production latency/cost benchmarks,
- model-registry governance, challenger evaluation and automated rollback,
- full-data (`SERIES_SAMPLE_PERCENT = 100`) scale validation.

# README-Ready Tables and Final Research Report

The report below is generated from the executed artifact values currently loaded in this notebook.  
No placeholder performance numbers are inserted.

In [ ]:
def fmt_num(x, digits: int = 4) -> str:
    try:
        value = float(x)
    except (TypeError, ValueError):
        return str(x)
    return "NA" if not np.isfinite(value) else f"{value:.{digits}f}"


def dataframe_markdown(df: pd.DataFrame, max_rows: Optional[int] = None) -> str:
    frame = df.copy()
    if max_rows is not None:
        frame = frame.head(max_rows)
    return frame.to_markdown(index=False)


def safe_pct(value: float) -> str:
    return "NA" if not np.isfinite(value) else f"{value:.2f}%"


executed_ablations = ablation_results.loc[ablation_results["evidence_status"].isin(["EXECUTED", "PARTIAL"])].copy()
positive_complexity = executed_ablations.loc[pd.to_numeric(executed_ablations["benefit_of_full_system_pct"], errors="coerce") > 0]
negative_complexity = executed_ablations.loc[pd.to_numeric(executed_ablations["benefit_of_full_system_pct"], errors="coerce") < 0]

monitoring_sentence = (
    f"The latest monitoring state was **{monitor_status}** with decision **{monitor_decision}**: "
    f"{latest_monitoring['reason']}"
)

causal_sentence = (
    f"The primary cross-fitted AIPW ATE was {fmt_num(aipw['ATE'])} "
    f"(95% CI {fmt_num(aipw['CI_Lower'])} to {fmt_num(aipw['CI_Upper'])}); "
    f"ATT was {fmt_num(att_main['ATT'])} "
    f"(95% CI {fmt_num(att_main['CI_Lower'])} to {fmt_num(att_main['CI_Upper'])}). "
    f"The sensitivity ATE range was {fmt_num(sens_min)} to {fmt_num(sens_max)}. "
    f"Placebo concern={placebo_concern}."
)

hierarchy_sentence = (
    f"The best reconciled method by overall WAPE was **{best_rec['method']}** "
    f"with WAPE {fmt_num(best_rec['WAPE'])} versus {fmt_num(unrec['WAPE'])} unreconciled. "
    f"Normalized coherence error changed from {fmt_num(unrec['normalized_coherence_error'], 6)} "
    f"to {fmt_num(best_rec['normalized_coherence_error'], 6)}."
)

inventory_sentence = (
    f"The lowest realized mean-cost inventory policy was **{best_inventory['policy']}**. "
    f"Relative to Historical Average, its realized mean-cost change was "
    f"{safe_pct(inventory_cost_gain_pct)} reduction when positive. "
    "These monetary results use Notebook 05's configurable simulation assumptions."
)

complexity_lines = []
for r in ablation_results.itertuples(index=False):
    benefit = getattr(r, "benefit_of_full_system_pct")
    benefit_text = "NA" if not np.isfinite(benefit) else f"{benefit:+.2f}%"
    complexity_lines.append(
        f"- **{r.component_removed}:** {r.evidence_status}; primary metric={r.primary_metric}; "
        f"benefit of full system={benefit_text}. {r.interpretation}"
    )

report = f"""# DemandPulse — Final Research Findings

## Executive Summary

DemandPulse was evaluated as an integrated research system rather than as a single forecasting model. The evidence spans point accuracy, uncertainty calibration, observational causal estimation, hierarchical coherence, inventory simulation and production-style monitoring.

- **Point forecasting:** LightGBM WAPE was {fmt_num(point_lgbm['WAPE'])} versus {fmt_num(point_seasonal['WAPE'])} for Seasonal Naive, corresponding to a relative WAPE improvement of {safe_pct(point_wape_gain_pct)} when positive.
- **Probabilistic forecasting:** raw P10–P90 coverage was {raw_prob['coverage']:.2%}; conformal coverage was {conf_prob['coverage']:.2%}. Mean interval width changed from {fmt_num(raw_prob['mean_width'])} to {fmt_num(conf_prob['mean_width'])}.
- **Causal evidence:** {causal_sentence}
- **Hierarchy:** {hierarchy_sentence}
- **Decision layer:** {inventory_sentence}
- **Monitoring:** {monitoring_sentence}

No composite “overall accuracy score” is reported because weighting forecasting error, causal credibility, business simulation and monitoring status into one number would be arbitrary.

## Research Questions

1. Do advanced global forecasts improve on simple seasonal baselines?
2. Are P10/P50/P90 forecasts and conformal intervals decision-useful?
3. Is promotion-response evidence robust enough to influence decisions?
4. Does reconciliation improve coherence without unacceptable accuracy loss?
5. Do forecast-driven inventory policies improve simulated costs/service outcomes?
6. Does the monitoring design detect drift, degradation and calibration failure without triggering unnecessary retraining?
7. Which added system components demonstrate measurable incremental value?

## Methodology

The final review consumed only saved outputs from DemandPulse Notebooks 01–06. It did not retrain forecasting, causal or reconciliation models. Comparisons were retained within their original chronological evaluation scopes. Cross-stage metrics are displayed together for system review but are not treated as automatically comparable when their observation masks differ.

The causal review emphasizes AIPW/ATT confidence intervals, sensitivity analysis and placebo evidence. Inventory financial metrics are explicitly simulation quantities under configurable cost assumptions. Monitoring decisions preserve the project rule that drift alone is insufficient for automatic retraining.

## Results

### Point Forecasting

{dataframe_markdown(point_final.sort_values('WAPE'))}

### Probabilistic Forecasting

{dataframe_markdown(prob_final)}

### Causal Promotion Effects

{dataframe_markdown(ate)}

{dataframe_markdown(att)}

### Hierarchical Reconciliation

{dataframe_markdown(reconciliation.sort_values('WAPE'))}

### Inventory Policy Comparison

{dataframe_markdown(inventory.sort_values('realized_mean_cost'))}

### Monitoring Status

{dataframe_markdown(retraining.tail(10))}

## Business Impact

{dataframe_markdown(business_impact_summary)}

## Ablation-Style Complexity Review

{dataframe_markdown(ablation_results)}

{chr(10).join(complexity_lines)}

The promotion-feature removal row is intentionally marked `NOT_EVALUATED` because no prior matched forecasting experiment removed those features. Causal promotion estimates are not a substitute for a predictive feature ablation.

## Failure Analysis

{dataframe_markdown(failure_analysis) if not failure_analysis.empty else "No failure rows were triggered by the configured evidence checks."}

## Computational Cost & Scalability

{dataframe_markdown(model_footprint)}

Saved model count/size is measurable from artifacts. Standardized training time, serving latency, infrastructure cost and full-data throughput were not logged as comparable experiment outputs and are therefore left as evidence gaps.

## Limitations

- `sale_amount` is normalized rather than audited unit demand.
- Stockouts can censor observed sales and therefore distort both forecasting and causal interpretation.
- Probabilistic forecasts use selected conditional quantiles rather than a complete probability distribution.
- Conformal validity depends on calibration/exchangeability assumptions and can trade sharpness for coverage.
- Promotion-effect estimates are observational and require no-major-unmeasured-confounding and positivity assumptions.
- CATE rankings are model-based heterogeneity estimates, not ground-truth product responses.
- Hierarchical MinT-style reconciliation uses a diagonal variance approximation rather than full residual covariance.
- Inventory costs, prices and waste/service parameters are simulation assumptions unless replaced with real business values.
- The causal adjustment should not be added twice when forecasting features already encode planned promotional activity.
- Cold-start and heavily stockout-censored entities require dedicated treatment.
- Full production-scale runtime, serving latency and cloud-cost benchmarks are not established by the current saved artifacts.
- Results are retailer-specific and may not transfer to unrelated demand processes.
- The promotion-feature forecasting ablation has not yet been executed.

## Production Recommendations

1. Deploy first in shadow mode with the full data-quality, drift, accuracy and calibration monitoring suite active.
2. Serve P50 plus calibrated uncertainty bounds, not a point forecast alone.
3. Use reconciled forecasts for planning processes that require accounting coherence, while monitoring hierarchy-level error.
4. Keep causal uplift advisory and uncertainty-aware; block deterministic uplift when evidence is weak or falsification diagnostics are concerning.
5. Replace inventory simulation assumptions with real margin, shortage, waste, holding and service-level economics before claiming financial ROI.
6. Retain persistence-based retraining gates and do not retrain from drift alone.
7. Add model registry/version lineage, feature-contract validation, shadow/challenger comparison and rollback.
8. Run the missing matched no-promotion-feature forecasting ablation.
9. Benchmark the full-data pipeline for throughput, latency, memory and cloud cost before production capacity planning.
10. Define explicit acceptance thresholds jointly with business owners for forecast error, interval coverage, service level and stockout risk.

## Final Architecture

```mermaid
flowchart TD
    A[Cloud Dataset] --> B[Data Validation]
    B --> C[Feature Pipeline]
    C --> D[Forecasting]
    D --> D1[Point]
    D --> D2[Probabilistic]
    D --> D3[Hierarchical]
    D1 --> E[Causal Promotion Engine]
    D2 --> E
    D3 --> E
    E --> F[Decision Optimization]
    F --> G[API]
    G --> H[Monitoring]
    H --> I[Retraining]
    I --> C
```

## Future Research

- probabilistic hierarchical reconciliation,
- latent-demand/lost-sales modeling under stockouts,
- stronger causal identification or randomized promotion experiments,
- cold-start forecasting,
- decision optimization with real service-level and profit economics,
- multi-dimensional grouped hierarchies,
- full-data scalability benchmarks,
- production API and feature-store architecture,
- governed automated retraining with challenger/shadow deployment.

---
Generated only from saved DemandPulse experiment artifacts. Missing experiments are reported as missing rather than fabricated.
"""

OUTPUT_PATHS["final_research_findings"].write_text(report, encoding="utf-8")
print(f"Wrote {OUTPUT_PATHS['final_research_findings']}")

# Save Final Tables

In [ ]:
final_project_scorecard.to_csv(OUTPUT_PATHS["final_project_scorecard"], index=False)
final_model_comparison.to_csv(OUTPUT_PATHS["final_model_comparison"], index=False)
business_impact_summary.to_csv(OUTPUT_PATHS["business_impact_summary"], index=False)
ablation_results.to_csv(OUTPUT_PATHS["ablation_results"], index=False)

for name, path in OUTPUT_PATHS.items():
    print(name, "->", path)

# Final Artifact & Consistency Audit

In [ ]:
final_audit = pd.DataFrame([
    {
        "artifact": name,
        "path": str(path),
        "exists": path.exists(),
        "size_bytes": path.stat().st_size if path.exists() else 0,
    }
    for name, path in OUTPUT_PATHS.items()
])
display(final_audit)

assert final_audit["exists"].all(), "One or more required final-evaluation outputs are missing."
assert (final_audit["size_bytes"] > 0).all(), "One or more final-evaluation outputs are empty."

# No fake metric rule.
assert ablation_results.loc[
    ablation_results["evidence_status"].eq("NOT_EVALUATED"),
    ["full_value", "ablated_value", "benefit_of_full_system_pct"]
].isna().all().all(), "Unexecuted ablations must not contain fabricated numbers."

# Source separation.
assert all(Path(p).parent == FINAL_DIR for p in OUTPUT_PATHS.values()), "Final outputs must stay in reports/final/."
assert (SOURCE_TABLES_DIR / "final_model_comparison.csv") != OUTPUT_PATHS["final_model_comparison"]

# Core evidence sanity.
assert np.isfinite(float(point_lgbm["WAPE"]))
assert 0 <= float(raw_prob["coverage"]) <= 1
assert 0 <= float(conf_prob["coverage"]) <= 1
assert np.isfinite(float(aipw["ATE"]))
assert np.isfinite(float(att_main["ATT"]))
assert np.isfinite(float(unrec["WAPE"]))
assert np.isfinite(float(best_rec["WAPE"]))
assert np.isfinite(float(hist_policy["realized_mean_cost"]))

print("Final consistency audit passed.")
print("Notebook 07 complete.")
print()
print("Saved:")
for path in OUTPUT_PATHS.values():
    print("✓", path)

# Final Review Conclusion

DemandPulse should be judged as a **stack of evidence** rather than by one headline metric:

- point accuracy establishes predictive value,
- probabilistic calibration establishes uncertainty quality,
- causal analysis estimates promotion-response evidence under explicit assumptions,
- hierarchical reconciliation enforces organizational consistency,
- inventory simulation tests decision consequences under configurable economics,
- monitoring determines whether the deployed system remains trustworthy over time.

The final production decision should be conditional on the actual values generated above, the latest monitoring status, the causal diagnostics, and replacement of simulation-only business costs with validated operational inputs.